In [1]:
!pip install transformers accelerate albumentations opencv-python pandas matplotlib scikit-learn pillow tqdm -q

In [4]:
import os
import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from transformers import SegformerForSemanticSegmentation

import albumentations as A
from albumentations.pytorch import ToTensorV2


PROJECT_DIR = r"C:\Users\sujal\OneDrive\Documents\praccodes\solar-filament-project"

IMAGE_DIR = os.path.join(
    PROJECT_DIR,
    "dataset",
    "kaggle",
    "train",
    "images"
)

MASK_DIR = os.path.join(
    PROJECT_DIR,
    "dataset",
    "masks"
)

CHECKPOINT_DIR = os.path.join(
    PROJECT_DIR,
    "checkpoints"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "outputs"
)


os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


IMAGE_SIZE = 512

BATCH_SIZE = 1

NUM_WORKERS = 0

EPOCHS = 20

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

SEED = 42

MODEL_NAME = "nvidia/mit-b0"


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


print("Device:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

print()
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Model:", MODEL_NAME)

Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.0 GB

Image size: 512
Batch size: 1
Epochs: 20
Learning rate: 0.0001
Model: nvidia/mit-b0


In [5]:
image_files = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(
        (
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".tif",
            ".tiff"
        )
    )
])


mask_files = sorted([
    f for f in os.listdir(MASK_DIR)
    if f.lower().endswith(".png")
])


image_basenames = {
    os.path.splitext(f)[0]
    for f in image_files
}


mask_basenames = {
    os.path.splitext(f)[0]
    for f in mask_files
}


matched = sorted(
    image_basenames &
    mask_basenames
)


missing_masks = sorted(
    image_basenames -
    mask_basenames
)


print(
    "Images:",
    len(image_files)
)

print(
    "Masks:",
    len(mask_files)
)

print(
    "Matched pairs:",
    len(matched)
)

print(
    "Images without masks:",
    len(missing_masks)
)


if missing_masks:

    print()
    print("Missing masks:")

    for name in missing_masks[:20]:
        print(name)


assert len(matched) > 0

assert len(missing_masks) == 0

print()
print("Dataset verification passed.")

Images: 707
Masks: 707
Matched pairs: 707
Images without masks: 0

Dataset verification passed.
